# Contextualized Ethomics: What Does a Hungry Fly Look Like?

The challenge: circuit manipulations in neuroscience frequently produce behavioural changes. 
But which manipulations produce *genuine* hunger versus isolated behavioural artefacts?

**DESTRA** (Dimensionality-reduced ESTimation of Reference-Anchored behavioural state) 
answers this by comparing the full multidimensional behavioural signature of a manipulation 
against the natural hunger–satiety transition. If a serotonin manipulation produces 
“hunger-like” behaviour, the full 17-metric profile should match 24h starvation — not just 
one or two metrics.

→ Paper in *Science Advances* (in revision)

---

## The Espresso System

Espresso is a custom behavioural tracking platform developed in the Claridge-Chang lab 
(IMCB, A\*STAR) that simultaneously monitors:

- **Feeding** via capacitive liquid-level detection in 64-channel acrylic microfluidic chips
- **Locomotion** via IR-backlit arenas imaged at 25 fps

Across a typical 2-hour session: **~100 flies × 25 fps × 7,200 seconds** = ~18 million 
frames of behaviour. The resulting dataset encodes 17 behavioural metrics per fly, 
spanning feeding microstructure (volume, meal size, bout counts, latency, feed speed) 
and locomotion dynamics (walk speed, peri-feed speed ratios, falls, port occupancy).

The hardware was designed to make the *invisible* visible: feeding is largely 
inaccessible to direct observation at the scale needed for genetic screens. 
Espresso makes it systematic.

---

In [ ]:
import dabest
from dabest.forest_plot import forest_plot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.patches import Patch
from scipy.interpolate import interp1d

np.random.seed(7)

mpl.rcParams['figure.facecolor'] = '#FAF7F2'
mpl.rcParams['axes.facecolor']   = '#FAF7F2'
mpl.rcParams['font.family']      = 'serif'

INK_BLUE  = '#2C4A6E'
CORAL     = '#D4735A'
VERDIGRIS = '#5B8A7A'
OCHRE     = '#C8A96E'
WARM_GREY = '#8B8478'

## Behavioural Metrics Across Conditions

The data below are synthetic but structurally realistic: the 17 metrics from the real 
Espresso/esploco pipeline measured across three conditions — **Fed**, **24h Starved** 
(the reference “hunger” state), and a **Serotonin KD** (serotonergic neuron knockdown) 
whose behavioural profile we want to characterise. Does the knockdown produce genuine hunger?

In [ ]:
# Real vectorselection from the paper (Holophenotype5HT, 01_Setup.ipynb)
vectorselection = [
    'Volume', 'Feed Speed', 'Meal Size', 'Meal Duration',
    'Duration', 'Count', 'Height',
    'Food Port Occupancy', 'Ctrl Port Occupancy', 'Latency',
    'Speed', 'Prefeed Speed', 'Duringfeed Speed', 'Postfeed Speed',
    'Duringfeed Speed Ratio', 'Perifeed Speed Ratio', 'Falls',
]
n_metrics = len(vectorselection)
N_flies   = 40

# Effect directions for 24h starvation vs fed
starved_shifts = np.array([
     2.1,   # Volume ↑
     0.8,   # Feed Speed ↑
     1.4,   # Meal Size ↑
     1.0,   # Meal Duration ↑
     1.2,   # Duration ↑
     1.5,   # Count ↑
     0.3,   # Height (small change)
     1.8,   # Food Port Occupancy ↑
    -0.6,   # Ctrl Port Occupancy ↓
    -1.3,   # Latency ↓ (faster to first feed)
     1.0,   # Speed ↑
     1.1,   # Prefeed Speed ↑
     0.2,   # Duringfeed Speed (small change)
     0.4,   # Postfeed Speed ↑
     0.3,   # Duringfeed Speed Ratio
     0.9,   # Perifeed Speed Ratio ↑
     0.7,   # Falls ↑
])

# Serotonin KD: ~82% of starvation profile (genuine but attenuated hunger state)
sert_kd_shifts = starved_shifts * 0.82 + np.random.normal(0, 0.12, n_metrics)

def make_flies(shifts, noise=0.75):
    return shifts + np.random.normal(0, noise, size=(N_flies, n_metrics))

fed_data   = make_flies(np.zeros(n_metrics))
starv_data = make_flies(starved_shifts)
kd_data    = make_flies(sert_kd_shifts)

# Long-format DataFrame for dabest.load()
status_col = ['Fed'] * N_flies + ['24h Starved'] * N_flies + ['Serotonin KD'] * N_flies
df = pd.DataFrame({'Status': status_col})
for k, metric in enumerate(vectorselection):
    df[metric] = np.concatenate([fed_data[:, k], starv_data[:, k], kd_data[:, k]])

In [ ]:
# Mean difference from Fed: Starved vs Fed, KD vs Fed
diffs = {
    '24h Starved': starv_data.mean(axis=0) - fed_data.mean(axis=0),
    'Serotonin KD': kd_data.mean(axis=0)   - fed_data.mean(axis=0),
}
colors_map = {'24h Starved': CORAL, 'Serotonin KD': VERDIGRIS}

fig, axes = plt.subplots(1, 2, figsize=(11, 6), sharey=True)

for ax, (cond, vals) in zip(axes, diffs.items()):
    ax.barh(range(n_metrics), vals, color=colors_map[cond],
            alpha=0.72, edgecolor=INK_BLUE, linewidth=0.5, height=0.7)
    ax.axvline(0, color=INK_BLUE, linewidth=0.8, alpha=0.4)
    ax.set_title(cond, fontsize=11, color=INK_BLUE, pad=8, fontweight='bold')
    ax.set_xlabel('\u0394 from Fed (a.u.)', fontsize=9, color=WARM_GREY)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(WARM_GREY)
    ax.tick_params(colors=WARM_GREY, labelsize=8.5)

axes[0].set_yticks(range(n_metrics))
axes[0].set_yticklabels(vectorselection, fontsize=8.5)

fig.suptitle('Mean difference from Fed across 17 Espresso metrics',
             fontsize=12, color=INK_BLUE, y=1.01)
plt.tight_layout()
plt.show()

## DESTRA: Effect Size Estimation Across All Metrics

The key question is not whether each metric differs individually — it is whether the 
**profile of differences** matches the reference. DESTRA computes bootstrap Hedges\' g 
for each metric, then asks: does the manipulation\u2019s effect size profile fall within 
the 95% CI of the starvation reference?

Below: the coral ribbon is the 95% bootstrap CI for **24h starvation** Hedges\' g across 
all 17 metrics, sorted by effect magnitude (most affected left). The **Serotonin KD** 
forest plot is overlaid. A manipulation that tracks inside the ribbon across most metrics 
meets the DESTRA criterion for a genuine hunger state.

In [ ]:
# One Hedges' g contrast per metric
contrasts_starv = [
    dabest.load(df, x='Status', y=m, idx=('Fed', '24h Starved'))
    for m in vectorselection
]
contrasts_kd = [
    dabest.load(df, x='Status', y=m, idx=('Fed', 'Serotonin KD'))
    for m in vectorselection
]

# Sort metrics by starvation Hedges' g magnitude (most affected first)
g_starv  = np.array([c.hedges_g.results.difference[0] for c in contrasts_starv])
sort_idx = np.argsort(np.abs(g_starv))[::-1]

metrics_sorted         = [vectorselection[i] for i in sort_idx]
contrasts_starv_sorted = [contrasts_starv[i]  for i in sort_idx]
contrasts_kd_sorted    = [contrasts_kd[i]     for i in sort_idx]

# Starvation reference: BCa 95% CI per metric
g_hi = np.array([c.hedges_g.results.bca_high[0] for c in contrasts_starv_sorted])
g_lo = np.array([c.hedges_g.results.bca_low[0]  for c in contrasts_starv_sorted])
x    = np.arange(1, n_metrics + 1)

# ── Starvation ribbon background ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 3.8))
fig.patch.set_facecolor('#FAF7F2')
ax.set_facecolor('#FAF7F2')

xs = np.linspace(1, n_metrics, 300)
ax.fill_between(xs,
                interp1d(x, g_lo, kind='cubic')(xs),
                interp1d(x, g_hi, kind='cubic')(xs),
                alpha=0.18, color=CORAL)
ax.plot(xs, interp1d(x, g_hi, kind='cubic')(xs), '-', color=CORAL, alpha=0.3, lw=0.6)
ax.plot(xs, interp1d(x, g_lo, kind='cubic')(xs), '-', color=CORAL, alpha=0.3, lw=0.6)
ax.axhline(0, color=WARM_GREY, lw=0.6, alpha=0.5)

# ── Serotonin KD forest plot ──────────────────────────────────────────────────
forest_plot(
    contrasts_kd_sorted,
    labels=metrics_sorted,
    effect_size='hedges_g',
    title="DESTRA: does Serotonin KD match the 24h starvation profile?",
    ax=ax,
    contrast_alpha=0.45,
    marker_size=7,
    ylabel="Hedges' g",
)

ax.legend(handles=[
    Patch(color=CORAL,     alpha=0.35, label='24h Starved — 95% BCa CI (reference)'),
    Patch(color=VERDIGRIS, alpha=0.8,  label='Serotonin KD'),
], fontsize=9, framealpha=0.85, loc='lower right')
ax.set_ylim(-3.5, 3.5)
plt.tight_layout()
plt.show()

The Serotonin KD profile tracks inside the starvation reference ribbon across most metrics — 
increased feeding volume, shortened latency, elevated locomotion speed. A manipulation that 
only affected one or two metrics would cross outside the ribbon on those dimensions while 
remaining flat on others. The profile match, not any single metric, is the criterion.

---

## The Failure Mode This Catches

Many manipulations look successful when you measure one or two metrics. A genetic 
manipulation that increases food intake looks like “hunger” on a tube feeding assay. 
But does the locomotion change? Does the latency compress? Does the fly spend more time 
at the food port and less at the control port?

DESTRA asks: **does the full behavioural profile match the natural state?**

This is the same failure mode that haunts AI evaluation: a model can score well on a 
benchmark while failing at the underlying task. The solution in both cases is 
ground-truth-referenced, multidimensional evaluation.

The measurement framework is the same whether the system is a fruit fly or a language model. 
That is not a coincidence.